In [0]:

# SMARTCITY AI - SILVER LAYER INITIALIZATION

CATALOG = "workspace"
BRONZE = "smartcity_bronze"
SILVER = "smartcity_silver"

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER}"
)

print("SMARTCITY AI - SILVER ENVIRONMENT READY")
print(f"Catalog: {CATALOG}")
print(f"Bronze schema: {BRONZE}")
print(f"Silver schema: {SILVER}")


SMARTCITY AI - SILVER ENVIRONMENT READY
Catalog: workspace
Bronze schema: smartcity_bronze
Silver schema: smartcity_silver


In [0]:

# SMARTCITY AI - BRONZE SOURCE INSPECTION

expected = {
    "taxi_trips": [
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "trip_distance",
        "PULocationID",
        "DOLocationID"
    ],
    "taxi_zones": [
        "LocationID",
        "Borough",
        "Zone"
    ],
    "traffic_speeds": [
        "data_as_of",
        "speed",
        "link_id"
    ],
    "road_collisions": [
        "crash_date",
        "collision_id"
    ]
}

for table, required_columns in expected.items():

    df = spark.table(
        f"{CATALOG}.{BRONZE}.{table}"
    )

    column_types = dict(df.dtypes)

    print(f"\n{'=' * 55}")
    print(f"TABLE: {table}")
    print(f"TOTAL COLUMNS: {len(df.columns)}")

    for col in required_columns:
        dtype = column_types.get(col, "MISSING")
        print(f"  {col}: {dtype}")

    print(f"{'=' * 55}")

print("\nSILVER SOURCE INSPECTION COMPLETED!")



TABLE: taxi_trips
TOTAL COLUMNS: 22
  tpep_pickup_datetime: timestamp_ntz
  tpep_dropoff_datetime: timestamp_ntz
  trip_distance: double
  PULocationID: int
  DOLocationID: int

TABLE: taxi_zones
TOTAL COLUMNS: 6
  LocationID: string
  Borough: string
  Zone: string

TABLE: traffic_speeds
TOTAL COLUMNS: 15
  data_as_of: string
  speed: string
  link_id: string

TABLE: road_collisions
TOTAL COLUMNS: 31
  crash_date: string
  collision_id: string

SILVER SOURCE INSPECTION COMPLETED!


In [0]:

# ===============================================
# SMARTCITY AI - SILVER TAXI TRIPS ETL
# ===============================================

from pyspark.sql import functions as F

BRONZE_TABLE = "workspace.smartcity_bronze.taxi_trips"
SILVER_TABLE = "workspace.smartcity_silver.taxi_trips"

# 1. Read Bronze data
bronze_df = spark.table(BRONZE_TABLE)

required = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance",
    "PULocationID",
    "DOLocationID"
]

missing = set(required) - set(bronze_df.columns)
assert not missing, f"Missing columns: {missing}"

print("STEP 1: Reading Bronze taxi trips")

bronze_count = bronze_df.count()
print(f"Bronze records: {bronze_count:,}")

# 2. Calculate trip duration
df = bronze_df.withColumn(
    "trip_duration_minutes",
    F.round(
        (
            F.unix_timestamp("tpep_dropoff_datetime")
            - F.unix_timestamp("tpep_pickup_datetime")
        ) / 60.0,
        2
    )
)

# 3. Apply data quality rules
valid_condition = (
    F.col("tpep_pickup_datetime").isNotNull()
    & F.col("tpep_dropoff_datetime").isNotNull()
    & (F.col("tpep_pickup_datetime") >= F.lit("2025-01-01").cast("timestamp"))
    & (F.col("tpep_pickup_datetime") < F.lit("2025-02-01").cast("timestamp"))
    & F.col("trip_duration_minutes").between(1, 180)
    & F.col("trip_distance").between(0.1, 150)
    & F.col("PULocationID").between(1, 265)
    & F.col("DOLocationID").between(1, 265)
)

valid_df = df.filter(valid_condition)

# 4. Remove exact source-record duplicates
source_columns = [
    c for c in bronze_df.columns
    if c not in ("_source_file", "_ingested_at")
]

clean_df = valid_df.dropDuplicates(source_columns)

# 5. Enrich records for BI and AI
clean_df = (
    clean_df
    .withColumn(
        "pickup_date",
        F.to_date("tpep_pickup_datetime")
    )
    .withColumn(
        "pickup_hour",
        F.hour("tpep_pickup_datetime")
    )
    .withColumn(
        "pickup_day_of_week",
        F.dayofweek("tpep_pickup_datetime")
    )
    .withColumn(
        "_silver_processed_at",
        F.current_timestamp()
    )
)

# 6. Calculate validation statistics
valid_count = valid_df.count()
clean_count = clean_df.count()

invalid_count = bronze_count - valid_count
duplicate_count = valid_count - clean_count

assert clean_count > 0, "No valid trips found!"

print("\nSMARTCITY AI - SILVER QUALITY REPORT")
print("=" * 48)
print(f"Bronze records:    {bronze_count:,}")
print(f"Invalid records:   {invalid_count:,}")
print(f"Exact duplicates:  {duplicate_count:,}")
print(f"Clean records:     {clean_count:,}")
print(f"Retention rate:    {clean_count / bronze_count * 100:.2f}%")

# 7. Safely create the first Silver table
if spark.catalog.tableExists(SILVER_TABLE):
    raise RuntimeError(
        "Silver taxi_trips already exists. "
        "Stopped to prevent accidental overwrite."
    )

(
    clean_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(SILVER_TABLE)
)

# 8. Verify the saved table
saved_count = spark.table(SILVER_TABLE).count()

table_format = (
    spark.sql(f"DESCRIBE DETAIL {SILVER_TABLE}")
    .select("format")
    .first()[0]
)

assert saved_count == clean_count
assert table_format.lower() == "delta"

print("\nSILVER TABLE:", SILVER_TABLE)
print("SAVED ROWS:", saved_count)
print("FORMAT:", table_format)
print("\nSUCCESS: SILVER TAXI TRIPS CREATED AND VERIFIED!")


STEP 1: Reading Bronze taxi trips
Bronze records: 3,475,226

SMARTCITY AI - SILVER QUALITY REPORT
Bronze records:    3,475,226
Invalid records:   119,215
Exact duplicates:  0
Clean records:     3,356,011
Retention rate:    96.57%

SILVER TABLE: workspace.smartcity_silver.taxi_trips
SAVED ROWS: 3356011
FORMAT: delta

SUCCESS: SILVER TAXI TRIPS CREATED AND VERIFIED!


In [0]:

table = "workspace.smartcity_silver.taxi_trips"

print("Table already exists:", spark.catalog.tableExists(table))


Table already exists: False


In [0]:

# SMARTCITY AI - SILVER TAXI DATA QUALITY TEST

from pyspark.sql import functions as F

table = "workspace.smartcity_silver.taxi_trips"
df = spark.table(table)

# Every valid trip must satisfy these conditions
valid = (
    F.col("tpep_pickup_datetime").isNotNull()
    & F.col("tpep_dropoff_datetime").isNotNull()
    & F.col("pickup_date").between(
        "2025-01-01", "2025-01-31"
    )
    & F.col("trip_duration_minutes").between(1, 180)
    & F.col("trip_distance").between(0.1, 150)
    & F.col("PULocationID").between(1, 265)
    & F.col("DOLocationID").between(1, 265)
    & F.col("pickup_hour").between(0, 23)
    & F.col("pickup_day_of_week").between(1, 7)
)

# One aggregated Spark query for efficient verification
result = df.agg(
    F.count("*").alias("total_rows"),
    F.sum(
        F.when(
            ~F.coalesce(valid, F.lit(False)), 1
        ).otherwise(0)
    ).alias("invalid_rows"),
    F.min("pickup_date").alias("earliest_date"),
    F.max("pickup_date").alias("latest_date"),
    F.round(
        F.avg("trip_duration_minutes"), 2
    ).alias("avg_duration_minutes"),
    F.round(
        F.avg("trip_distance"), 2
    ).alias("avg_distance_miles")
).first().asDict()

print("SMARTCITY AI - SILVER QUALITY REPORT")
print("=" * 45)

for key, value in result.items():
    print(f"{key}: {value}")

assert result["total_rows"] == 3356011, (
    "Unexpected row count!"
)

assert result["invalid_rows"] == 0, (
    "Invalid records detected!"
)

print("\nALL SILVER TAXI QUALITY TESTS PASSED!")


SMARTCITY AI - SILVER QUALITY REPORT
total_rows: 3356011
invalid_rows: 0
earliest_date: 2025-01-01
latest_date: 2025-01-31
avg_duration_minutes: 14.74
avg_distance_miles: 3.2

ALL SILVER TAXI QUALITY TESTS PASSED!


In [0]:

# ==========================================
# SMARTCITY AI - SILVER TAXI ZONES
# ==========================================

from pyspark.sql import functions as F

BRONZE_TABLE = "workspace.smartcity_bronze.taxi_zones"
SILVER_TABLE = "workspace.smartcity_silver.taxi_zones"

# Protect any existing Silver data
if spark.catalog.tableExists(SILVER_TABLE):
    raise RuntimeError(
        "Silver taxi_zones already exists. "
        "Stop to prevent duplicate records."
    )

# 1. Read Bronze data
bronze_df = spark.table(BRONZE_TABLE)

required = {"LocationID", "Borough", "Zone"}
missing = required - set(bronze_df.columns)
assert not missing, f"Missing columns: {missing}"

bronze_count = bronze_df.count()

# 2. Normalize strings
def clean_text(column):
    value = F.trim(F.col(column))
    return F.when(F.length(value) > 0, value)

# 3. Convert data types
service_zone = (
    clean_text("service_zone")
    if "service_zone" in bronze_df.columns
    else F.lit(None).cast("string")
)

df = bronze_df.select(
    F.expr("try_cast(LocationID AS INT)").alias("location_id"),
    clean_text("Borough").alias("borough"),
    clean_text("Zone").alias("zone"),
    service_zone.alias("service_zone")
)

# 4. Remove invalid records
valid_df = df.filter(
    F.col("location_id").between(1, 265)
    & F.col("borough").isNotNull()
    & F.col("zone").isNotNull()
)

# 5. Remove exact duplicates
clean_df = valid_df.dropDuplicates(
    ["location_id", "borough", "zone", "service_zone"]
)

# 6. Ensure every location ID is unique
conflicting_ids = (
    clean_df.groupBy("location_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert conflicting_ids == 0, (
    f"{conflicting_ids} conflicting location IDs found!"
)

# 7. Add useful attributes
clean_df = (
    clean_df
    .withColumn(
        "is_special_zone",
        F.col("location_id").isin(264, 265)
    )
    .withColumn(
        "_silver_processed_at",
        F.current_timestamp()
    )
)

# 8. Quality report
valid_count = valid_df.count()
clean_count = clean_df.count()

print("SMARTCITY AI - TAXI ZONES QUALITY REPORT")
print("=" * 45)
print("Bronze records:", bronze_count)
print("Invalid records:", bronze_count - valid_count)
print("Exact duplicates:", valid_count - clean_count)
print("Clean records:", clean_count)
print("Conflicting location IDs:", conflicting_ids)

assert clean_count > 0
assert clean_count <= bronze_count

# 9. Write Delta table
clean_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(SILVER_TABLE)

# 10. Verify saved table
saved_count = spark.table(SILVER_TABLE).count()

table_format = spark.sql(
    f"DESCRIBE DETAIL {SILVER_TABLE}"
).select("format").first()[0]

assert saved_count == clean_count
assert table_format.lower() == "delta"

print("\nSAVED ROWS:", saved_count)
print("FORMAT:", table_format)
print("SUCCESS: SILVER TAXI ZONES CREATED!")


SMARTCITY AI - TAXI ZONES QUALITY REPORT
Bronze records: 265
Invalid records: 0
Exact duplicates: 0
Clean records: 265
Conflicting location IDs: 0

SAVED ROWS: 265
FORMAT: delta
SUCCESS: SILVER TAXI ZONES CREATED!


In [0]:

# ==========================================
# SMARTCITY AI - GEOGRAPHIC QUALITY TEST
# ==========================================

from pyspark.sql import functions as F

zones = spark.table(
    "workspace.smartcity_silver.taxi_zones"
)

trips = spark.table(
    "workspace.smartcity_silver.taxi_trips"
)

# 1. Check zone uniqueness and null IDs
zone_stats = zones.agg(
    F.count("*").alias("total_zones"),
    F.countDistinct("location_id").alias("unique_ids"),
    F.sum(
        F.when(
            F.col("location_id").isNull(), 1
        ).otherwise(0)
    ).alias("null_ids")
).first().asDict()

print("GEOGRAPHIC DIMENSION QUALITY")
print("=" * 45)

for key, value in zone_stats.items():
    print(f"{key}: {value}")

assert zone_stats["total_zones"] > 0
assert zone_stats["null_ids"] == 0
assert (
    zone_stats["total_zones"]
    == zone_stats["unique_ids"]
), "Duplicate location IDs found!"

# 2. Connect pickup and drop-off zones
mapped = (
    trips.alias("t")
    .join(
        F.broadcast(zones.alias("pu")),
        F.col("t.PULocationID")
        == F.col("pu.location_id"),
        "left"
    )
    .join(
        F.broadcast(zones.alias("do")),
        F.col("t.DOLocationID")
        == F.col("do.location_id"),
        "left"
    )
)

# 3. Count unmatched location IDs
join_stats = mapped.agg(
    F.count("*").alias("total_trips"),
    F.sum(
        F.when(
            F.col("pu.location_id").isNull(), 1
        ).otherwise(0)
    ).alias("unmatched_pickups"),
    F.sum(
        F.when(
            F.col("do.location_id").isNull(), 1
        ).otherwise(0)
    ).alias("unmatched_dropoffs")
).first().asDict()

print("\nTAXI TRIP GEOGRAPHIC VALIDATION")
print("=" * 45)

for key, value in join_stats.items():
    print(f"{key}: {value}")

# Ensure every trip matches a real lookup ID
assert join_stats["unmatched_pickups"] == 0, (
    "Some pickup IDs are missing from taxi_zones"
)

assert join_stats["unmatched_dropoffs"] == 0, (
    "Some drop-off IDs are missing from taxi_zones"
)

# 4. Display sample geographic mappings
print("\nSAMPLE NYC JOURNEYS")

mapped.select(
    F.col("t.PULocationID").alias("pickup_id"),
    F.col("pu.zone").alias("pickup_zone"),
    F.col("pu.borough").alias("pickup_borough"),
    F.col("t.DOLocationID").alias("dropoff_id"),
    F.col("do.zone").alias("dropoff_zone")
).show(10, truncate=False)

print("\nSUCCESS: GEOGRAPHIC QUALITY PASSED!")


GEOGRAPHIC DIMENSION QUALITY
total_zones: 265
unique_ids: 265
null_ids: 0

TAXI TRIP GEOGRAPHIC VALIDATION
total_trips: 3356011
unmatched_pickups: 0
unmatched_dropoffs: 0

SAMPLE NYC JOURNEYS
+---------+----------------------------+--------------+----------+-----------------------------+
|pickup_id|pickup_zone                 |pickup_borough|dropoff_id|dropoff_zone                 |
+---------+----------------------------+--------------+----------+-----------------------------+
|142      |Lincoln Square East         |Manhattan     |50        |Clinton West                 |
|239      |Upper West Side South       |Manhattan     |151       |Manhattan Valley             |
|140      |Lenox Hill East             |Manhattan     |140       |Lenox Hill East              |
|79       |East Village                |Manhattan     |246       |West Chelsea/Hudson Yards    |
|234      |Union Sq                    |Manhattan     |261       |World Trade Center           |
|231      |TriBeCa/Civic Center 

In [0]:

# ==============================================
# SMARTCITY AI - SILVER TRAFFIC SPEEDS ETL
# ==============================================

from pyspark.sql import functions as F

BRONZE_TABLE = "workspace.smartcity_bronze.traffic_speeds"
SILVER_TABLE = "workspace.smartcity_silver.traffic_speeds"

# 1. Protect any existing Silver table
if spark.catalog.tableExists(SILVER_TABLE):
    raise RuntimeError(
        "Silver traffic_speeds already exists. "
        "Stopped to prevent accidental duplication."
    )

# 2. Read Bronze
bronze_df = spark.table(BRONZE_TABLE)
bronze_count = bronze_df.count()

required = {"data_as_of", "speed", "link_id"}
missing = required - set(bronze_df.columns)

assert not missing, f"Missing columns: {missing}"

print("BRONZE RECORDS:", bronze_count)

# 3. Convert essential fields safely
df = (
    bronze_df
    .withColumn(
        "observation_time",
        F.expr("try_to_timestamp(data_as_of)")
    )
    .withColumn(
        "speed_mph",
        F.expr("try_cast(trim(speed) AS DOUBLE)")
    )
    .withColumn(
        "road_link_id",
        F.trim(F.col("link_id"))
    )
)

# Optional travel time conversion
if "travel_time" in df.columns:
    df = df.withColumn(
        "travel_time_seconds",
        F.expr("try_cast(trim(travel_time) AS DOUBLE)")
    )
else:
    df = df.withColumn(
        "travel_time_seconds",
        F.lit(None).cast("double")
    )

# Replace negative travel times with null
df = df.withColumn(
    "travel_time_seconds",
    F.when(
        F.col("travel_time_seconds") >= 0,
        F.col("travel_time_seconds")
    )
)

# 4. Filter invalid observations
valid_df = df.filter(
    F.col("observation_time").isNotNull()
    & F.col("road_link_id").isNotNull()
    & (F.length("road_link_id") > 0)
    & F.col("speed_mph").between(0, 120)
    & (
        (F.col("observation_time") >=
         F.lit("2025-01-01").cast("timestamp"))
        &
        (F.col("observation_time") <
         F.lit("2025-02-01").cast("timestamp"))
    )
)

# 5. Remove duplicate measurements
clean_df = valid_df.dropDuplicates([
    "road_link_id",
    "observation_time",
    "speed_mph",
    "travel_time_seconds"
])

# 6. Add useful BI features
clean_df = (
    clean_df
    .withColumn(
        "observation_date",
        F.to_date("observation_time")
    )
    .withColumn(
        "observation_hour",
        F.hour("observation_time")
    )
    .withColumn(
        "day_of_week",
        F.dayofweek("observation_time")
    )
    .withColumn(
        "is_zero_speed",
        F.col("speed_mph") == 0
    )
    .withColumn(
        "_silver_processed_at",
        F.current_timestamp()
    )
)

# 7. Data quality statistics
valid_count = valid_df.count()
clean_count = clean_df.count()

print("\nSMARTCITY AI - TRAFFIC QUALITY REPORT")
print("=" * 45)
print("Bronze records:", bronze_count)
print("Invalid records:", bronze_count - valid_count)
print("Duplicates removed:", valid_count - clean_count)
print("Clean records:", clean_count)

assert clean_count > 0, (
    "No valid traffic observations found. "
    "Check timestamp parsing and source dates."
)

# 8. Create managed Delta table
(
    clean_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(SILVER_TABLE)
)

# 9. Verify table
saved_count = spark.table(SILVER_TABLE).count()

table_format = spark.sql(
    f"DESCRIBE DETAIL {SILVER_TABLE}"
).select("format").first()[0]

assert saved_count == clean_count
assert table_format.lower() == "delta"

print("\nSAVED ROWS:", saved_count)
print("FORMAT:", table_format)
print("SUCCESS: SILVER TRAFFIC SPEEDS CREATED!")


BRONZE RECORDS: 5000

SMARTCITY AI - TRAFFIC QUALITY REPORT
Bronze records: 5000
Invalid records: 0
Duplicates removed: 0
Clean records: 5000

SAVED ROWS: 5000
FORMAT: delta
SUCCESS: SILVER TRAFFIC SPEEDS CREATED!


In [0]:

# ===========================================
# SMARTCITY AI - TRAFFIC QUALITY VERIFICATION
# ===========================================

from pyspark.sql import functions as F

TABLE = "workspace.smartcity_silver.traffic_speeds"
df = spark.table(TABLE)

# 1. Required columns
required = {
    "road_link_id",
    "observation_time",
    "observation_hour",
    "speed_mph",
    "travel_time_seconds"
}

missing = required - set(df.columns)
assert not missing, f"Missing columns: {missing}"

# 2. Define valid measurements
valid = (
    F.col("road_link_id").isNotNull()
    & (F.length(F.trim("road_link_id")) > 0)
    & F.col("observation_time").isNotNull()
    & F.to_date("observation_time").between(
        "2025-01-01", "2025-01-31"
    )
    & F.col("observation_hour").between(0, 23)
    & F.col("speed_mph").isNotNull()
    & ~F.isnan("speed_mph")
    & F.col("speed_mph").between(0, 120)
    & (
        F.col("travel_time_seconds").isNull()
        | (F.col("travel_time_seconds") >= 0)
    )
)

# 3. Calculate quality and speed metrics
stats = df.agg(
    F.count("*").alias("total_records"),
    F.sum(
        F.when(
            ~F.coalesce(valid, F.lit(False)), 1
        ).otherwise(0)
    ).alias("invalid_records"),
    F.countDistinct("road_link_id").alias("road_links"),
    F.countDistinct(
        F.to_date("observation_time")
    ).alias("days_covered"),
    F.min("observation_time").alias("first_observation"),
    F.max("observation_time").alias("last_observation"),
    F.round(F.avg("speed_mph"), 2).alias("avg_speed_mph"),
    F.expr(
        "percentile_approx(speed_mph, 0.5)"
    ).alias("median_speed_mph"),
    F.min("speed_mph").alias("min_speed_mph"),
    F.max("speed_mph").alias("max_speed_mph"),
    F.sum(
        F.when(F.col("speed_mph") == 0, 1).otherwise(0)
    ).alias("zero_speed_records")
).first().asDict()

# 4. Verify no duplicate measurements
duplicates = (
    df.groupBy(
        "road_link_id",
        "observation_time",
        "speed_mph",
        "travel_time_seconds"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# 5. Print results
print("SMARTCITY AI - TRAFFIC QUALITY REPORT")
print("=" * 50)

for key, value in stats.items():
    print(f"{key}: {value}")

print("duplicate_measurement_groups:", duplicates)

assert stats["total_records"] > 0
assert stats["invalid_records"] == 0
assert duplicates == 0

print("\nSUCCESS: ALL TRAFFIC QUALITY TESTS PASSED!")


SMARTCITY AI - TRAFFIC QUALITY REPORT
total_records: 5000
invalid_records: 0
road_links: 122
days_covered: 1
first_observation: 2025-01-01 00:03:03
last_observation: 2025-01-01 03:23:08
avg_speed_mph: 32.88
median_speed_mph: 42.25
min_speed_mph: 0.0
max_speed_mph: 68.35
zero_speed_records: 1131
duplicate_measurement_groups: 0

SUCCESS: ALL TRAFFIC QUALITY TESTS PASSED!


In [0]:

# SMARTCITY AI - HOURLY TRAFFIC ANALYSIS

hourly_traffic = (
    df.groupBy("observation_hour")
    .agg(
        F.count("*").alias("measurements"),
        F.round(
            F.avg("speed_mph"), 2
        ).alias("avg_speed_mph"),
        F.countDistinct(
            "road_link_id"
        ).alias("road_links")
    )
    .orderBy("observation_hour")
)

print("TRAFFIC SPEED BY HOUR")
display(hourly_traffic)


TRAFFIC SPEED BY HOUR


observation_hour,measurements,avg_speed_mph,road_links
0,1464,33.12,122
1,1464,31.85,122
2,1464,33.19,122
3,608,34.02,122


In [0]:

from pyspark.sql import functions as F

df = spark.table(
    "workspace.smartcity_bronze.road_collisions"
)

print("=== ORIGINAL DATA SAMPLE ===")

df.select(
    "collision_id",
    "crash_date",
    "crash_time"
).show(10, truncate=False)

print("\n=== DATE DISTRIBUTION ===")

df.groupBy(
    F.substring("crash_date", 1, 7).alias("date_prefix")
).count().orderBy(F.desc("count")).show(15, False)

print("\n=== CONVERSION DIAGNOSTICS ===")

test = df.select(
    F.expr(
        "try_cast(trim(collision_id) AS BIGINT)"
    ).alias("parsed_id"),

    F.expr(
        "try_cast(substr(crash_date, 1, 10) AS DATE)"
    ).alias("parsed_date")
)

test.agg(
    F.count("*").alias("total"),
    F.count("parsed_id").alias("valid_ids"),
    F.count("parsed_date").alias("valid_dates"),
    F.sum(
        F.when(
            F.col("parsed_date").between(
                "2025-01-01", "2025-01-31"
            ), 1
        ).otherwise(0)
    ).alias("january_2025_records")
).show()


=== ORIGINAL DATA SAMPLE ===
+------------+-----------------------+-------------+
|collision_id|crash_date             |crash_time   |
+------------+-----------------------+-------------+
|NULL        |2025-01-01T00:00:00.000|21:37        |
|NULL        |NULL                   |             |
|NULL        |(40.799984             | -73.94485)" |
|NULL        |2025-01-01T00:00:00.000|11:20        |
|NULL        |NULL                   |             |
|NULL        |(40.66658              | -73.939674)"|
|NULL        |2025-01-01T00:00:00.000|09:21        |
|NULL        |NULL                   |             |
|NULL        |(0.0                   | 0.0)"       |
|NULL        |2025-01-01T00:00:00.000|02:50        |
+------------+-----------------------+-------------+
only showing top 10 rows

=== DATE DISTRIBUTION ===
+-----------+-----+
|date_prefix|count|
+-----------+-----+
|2025-01    |6482 |
|NULL       |6404 |
|(0.0       |138  |
|(40.666    |70   |
|(40.665    |49   |
|(40.679    |45  

In [0]:

from pyspark.sql import functions as F

RAW_PATH = (
    "/Volumes/workspace/smartcity_bronze/"
    "smartcity_raw/nyc_crashes_2025-01.csv"
)

# Read original CSV with multiline support
recovered_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .load(RAW_PATH)
)

required = {"collision_id", "crash_date"}
assert required.issubset(set(recovered_df.columns))

# Inspect repaired records
recovered_df.select(
    "collision_id",
    "crash_date",
    "crash_time"
).show(5, truncate=False)

# Verify conversions
check = recovered_df.select(
    F.expr(
        "try_cast(trim(collision_id) AS BIGINT)"
    ).alias("id"),
    F.expr(
        "try_cast(substr(trim(crash_date),1,10) AS DATE)"
    ).alias("date")
)

stats = check.agg(
    F.count("*").alias("total"),
    F.count("id").alias("valid_ids"),
    F.count("date").alias("valid_dates"),
    F.sum(
        F.when(
            F.col("date").between(
                "2025-01-01", "2025-01-31"
            ), 1
        ).otherwise(0)
    ).alias("january_records")
).first().asDict()

print("CSV REPAIR DIAGNOSTICS")
for key, value in stats.items():
    print(f"{key}: {value}")

assert stats["total"] > 0
assert stats["valid_ids"] / stats["total"] >= 0.95
assert stats["january_records"] / stats["total"] >= 0.95

print("SUCCESS: CSV PARSING RECOVERED!")


+--------------------------------+-----------------------+----------+
|collision_id                    |crash_date             |crash_time|
+--------------------------------+-----------------------+----------+
|4c25d48db62d3133aefdbf68b5d30b9f|2025-01-01T00:00:00.000|21:37     |
|5b03fc0580bb52eb947ff9b2e32307d5|2025-01-01T00:00:00.000|11:20     |
|7f17fad9df3819d1d518930c56332d0e|2025-01-01T00:00:00.000|09:21     |
|f9f451da4441f9a75bd525f50dd5149f|2025-01-01T00:00:00.000|02:50     |
|2f9e9a23cc1df6085968a54f3cbac1b9|2025-01-01T00:00:00.000|06:00     |
+--------------------------------+-----------------------+----------+
only showing top 5 rows
CSV REPAIR DIAGNOSTICS
total: 6482
valid_ids: 0
valid_dates: 6482
january_records: 6482


---------------------------------------------------------------------------
AssertionError                            Traceback (most recent call last)
File <command-8411031912141945>, line 58
     55     print(f"{key}: {value}")
     57 assert stats["total"] > 0
---> 58 assert stats["valid_ids"] / stats["total"] >= 0.95
     59 assert stats["january_records"] / stats["total"] >= 0.95
     61 print("SUCCESS: CSV PARSING RECOVERED!")

AssertionError: 

In [0]:

# SMARTCITY AI - CORRECTED ROAD COLLISIONS ETL
# Keeps collision IDs as strings
# Preserves original Bronze and existing Silver data

from pyspark.sql import functions as F

RAW_PATH = (
    "/Volumes/workspace/smartcity_bronze/"
    "smartcity_raw/nyc_crashes_2025-01.csv"
)

FIXED_BRONZE = (
    "workspace.smartcity_bronze.road_collisions_reparsed"
)
SILVER = (
    "workspace.smartcity_silver.road_collisions"
)

# Safety: never overwrite existing tables
for table in [FIXED_BRONZE, SILVER]:
    if spark.catalog.tableExists(table):
        raise RuntimeError(
            f"Table already exists: {table}. "
            "Stop to prevent accidental changes."
        )

# 1. Read original CSV correctly
raw = (
    spark.read
    .option("header", "true")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .csv(RAW_PATH)
)

# 2. Convert types, preserve string IDs
df = (
    raw
    .withColumn(
        "collision_id",
        F.trim(F.col("collision_id"))
    )
    .withColumn(
        "crash_date",
        F.expr(
            "try_cast(substr(trim(crash_date), 1, 10) AS DATE)"
        )
    )
)

# 3. Keep valid January records
df = df.filter(
    (F.length("collision_id") > 0)
    & F.col("crash_date").between(
        "2025-01-01", "2025-01-31"
    )
)

# 4. Check duplicate collision IDs
duplicates = (
    df.groupBy("collision_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert duplicates == 0, (
    f"{duplicates} duplicate ID groups found. "
    "Inspect before writing."
)

# 5. Clean geographic coordinates
for field in ["latitude", "longitude"]:
    if field in df.columns:
        df = df.withColumn(
            field + "_clean",
            F.expr(
                f"try_cast(trim({field}) AS DOUBLE)"
            )
        )
    else:
        df = df.withColumn(
            field + "_clean",
            F.lit(None).cast("double")
        )

df = df.withColumn(
    "is_mappable",
    F.coalesce(
        F.col("latitude_clean").between(40.4, 41.1)
        & F.col("longitude_clean").between(-74.4, -73.6),
        F.lit(False)
    )
)

# 6. Clean injury and fatality columns
metrics = {
    "number_of_persons_injured": "persons_injured",
    "number_of_persons_killed": "persons_killed",
    "number_of_pedestrians_injured": "pedestrians_injured",
    "number_of_cyclist_injured": "cyclists_injured",
    "number_of_motorist_injured": "motorists_injured"
}

for source, target in metrics.items():
    if source in df.columns:
        value = F.expr(
            f"try_cast(trim({source}) AS INT)"
        )
        df = df.withColumn(
            target,
            F.when(value >= 0, value)
        )
    else:
        df = df.withColumn(
            target,
            F.lit(None).cast("int")
        )

# 7. Add analytical features
if "crash_time" in df.columns:
    df = df.withColumn(
        "crash_hour",
        F.when(
            F.trim("crash_time").rlike(
                r"^(?:[01]?\d|2[0-3]):[0-5]\d$"
            ),
            F.expr(
                "try_cast(split(trim(crash_time), ':')[0] AS INT)"
            )
        )
    )
else:
    df = df.withColumn(
        "crash_hour",
        F.lit(None).cast("int")
    )

df = (
    df
    .withColumn(
        "borough_clean",
        F.upper(F.trim("borough"))
    )
    .withColumn(
        "crash_day_of_week",
        F.dayofweek("crash_date")
    )
    .withColumn(
        "has_injuries",
        F.coalesce(
            F.col("persons_injured") > 0,
            F.lit(False)
        )
    )
    .withColumn(
        "has_fatalities",
        F.coalesce(
            F.col("persons_killed") > 0,
            F.lit(False)
        )
    )
    .withColumn(
        "_silver_processed_at",
        F.current_timestamp()
    )
)

# 8. Validate before writing
clean_count = df.count()

print("CLEAN COLLISIONS:", clean_count)
print("DUPLICATE ID GROUPS:", duplicates)

assert clean_count > 0

# 9. Save corrected source separately
(
    raw
    .withColumn("_source_file", F.lit("nyc_crashes_2025-01.csv"))
    .withColumn("_ingested_at", F.current_timestamp())
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(FIXED_BRONZE)
)

# 10. Create Silver table
(
    df.write
    .format("delta")
    .mode("append")
    .saveAsTable(SILVER)
)

# 11. Final validation
saved = spark.table(SILVER).count()

fmt = (
    spark.sql(f"DESCRIBE DETAIL {SILVER}")
    .select("format")
    .first()[0]
)

assert saved == clean_count
assert fmt.lower() == "delta"

print("\nSUCCESS: ROAD COLLISIONS SILVER VERIFIED!")
print("ROWS:", saved)
print("FORMAT:", fmt)


CLEAN COLLISIONS: 6482
DUPLICATE ID GROUPS: 0

SUCCESS: ROAD COLLISIONS SILVER VERIFIED!
ROWS: 6482
FORMAT: delta


In [0]:

# SMARTCITY AI - ROAD SAFETY QUALITY CHECK

from pyspark.sql import functions as F

TABLE = "workspace.smartcity_silver.road_collisions"
SOURCE = "workspace.smartcity_bronze.road_collisions_reparsed"

df = spark.table(TABLE)

required = {
    "collision_id", "crash_date", "crash_hour",
    "latitude_clean", "longitude_clean",
    "is_mappable", "persons_injured", "persons_killed"
}

missing = required - set(df.columns)
assert not missing, f"Missing columns: {missing}"

# Quality rules
valid_id = (
    F.col("collision_id").isNotNull()
    & (F.length(F.trim("collision_id")) > 0)
)

valid_date = F.col("crash_date").between(
    "2025-01-01", "2025-01-31"
)

valid_hour = (
    F.col("crash_hour").isNull()
    | F.col("crash_hour").between(0, 23)
)

valid_casualties = (
    (F.col("persons_injured").isNull()
     | (F.col("persons_injured") >= 0))
    &
    (F.col("persons_killed").isNull()
     | (F.col("persons_killed") >= 0))
)

expected_mappable = F.coalesce(
    F.col("latitude_clean").between(40.4, 41.1)
    & F.col("longitude_clean").between(-74.4, -73.6),
    F.lit(False)
)

def invalid_count(condition):
    return F.sum(
        F.when(
            ~F.coalesce(condition, F.lit(False)), 1
        ).otherwise(0)
    )

stats = df.agg(
    F.count("*").alias("total_collisions"),
    F.countDistinct("collision_id").alias("unique_ids"),
    invalid_count(valid_id).alias("invalid_ids"),
    invalid_count(valid_date).alias("invalid_dates"),
    invalid_count(valid_hour).alias("invalid_hours"),
    invalid_count(valid_casualties).alias("invalid_casualties"),
    invalid_count(
        F.col("is_mappable").eqNullSafe(expected_mappable)
    ).alias("incorrect_map_flags"),
    F.sum(
        F.when(F.col("is_mappable"), 1).otherwise(0)
    ).alias("mappable_collisions"),
    F.sum(
        F.when(
            F.col("latitude_clean").isNull()
            | F.col("longitude_clean").isNull(),
            1
        ).otherwise(0)
    ).alias("missing_coordinates"),
    F.sum(
        F.when(
            F.col("persons_injured").isNull(), 1
        ).otherwise(0)
    ).alias("missing_injury_counts"),
    F.sum(
        F.when(
            F.col("persons_killed").isNull(), 1
        ).otherwise(0)
    ).alias("missing_fatality_counts")
).first().asDict()

print("SMARTCITY AI - ROAD SAFETY QUALITY REPORT")
print("=" * 50)

for key, value in stats.items():
    print(f"{key}: {value}")

source_count = spark.table(SOURCE).count()
print("Reparsed Bronze records:", source_count)

# Final assertions
assert stats["total_collisions"] > 0
assert stats["total_collisions"] <= source_count
assert stats["unique_ids"] == stats["total_collisions"]

for name in [
    "invalid_ids",
    "invalid_dates",
    "invalid_hours",
    "invalid_casualties",
    "incorrect_map_flags"
]:
    assert stats[name] == 0, f"FAILED: {name}"

print("\nSUCCESS: ALL ROAD SAFETY QUALITY TESTS PASSED!")


SMARTCITY AI - ROAD SAFETY QUALITY REPORT
total_collisions: 6482
unique_ids: 6482
invalid_ids: 0
invalid_dates: 0
invalid_hours: 0
invalid_casualties: 0
incorrect_map_flags: 0
mappable_collisions: 6266
missing_coordinates: 78
missing_injury_counts: 0
missing_fatality_counts: 0
Reparsed Bronze records: 6482

SUCCESS: ALL ROAD SAFETY QUALITY TESTS PASSED!


In [0]:

# ===============================================
# SMARTCITY AI - ROAD SAFETY QUALITY VALIDATION
# ===============================================

from pyspark.sql import functions as F

SILVER = "workspace.smartcity_silver.road_collisions"
SOURCE = "workspace.smartcity_bronze.road_collisions_reparsed"

df = spark.table(SILVER)

# 1. Check required fields
required = {
    "collision_id", "crash_date",
    "crash_hour", "latitude_clean",
    "longitude_clean", "is_mappable",
    "persons_injured", "persons_killed"
}

missing = required - set(df.columns)
assert not missing, f"Missing columns: {missing}"

# 2. Define validity checks
valid_id = (
    F.col("collision_id").isNotNull()
    & (F.length(F.trim("collision_id")) > 0)
)

valid_date = F.coalesce(
    F.col("crash_date").between(
        "2025-01-01", "2025-01-31"
    ),
    F.lit(False)
)

valid_hour = (
    F.col("crash_hour").isNull()
    | F.col("crash_hour").between(0, 23)
)

valid_casualties = (
    (F.col("persons_injured").isNull()
     | (F.col("persons_injured") >= 0))
    &
    (F.col("persons_killed").isNull()
     | (F.col("persons_killed") >= 0))
)

valid_location = (
    F.col("latitude_clean").between(40.4, 41.1)
    & F.col("longitude_clean").between(-74.4, -73.6)
)

# 3. Calculate quality indicators
stats = df.agg(
    F.count("*").alias("total_collisions"),

    F.countDistinct("collision_id").alias("unique_ids"),

    F.sum(F.when(~valid_id, 1).otherwise(0)).alias(
        "invalid_ids"
    ),

    F.sum(F.when(~valid_date, 1).otherwise(0)).alias(
        "invalid_dates"
    ),

    F.sum(F.when(
        ~F.coalesce(valid_hour, F.lit(False)), 1
    ).otherwise(0)).alias("invalid_hours"),

    F.sum(F.when(
        ~F.coalesce(valid_casualties, F.lit(False)), 1
    ).otherwise(0)).alias("invalid_casualties"),

    F.sum(F.when(
        F.col("is_mappable") == True, 1
    ).otherwise(0)).alias("mappable_collisions"),

    F.sum(F.when(
        F.col("latitude_clean").isNull()
        | F.col("longitude_clean").isNull(), 1
    ).otherwise(0)).alias("missing_coordinates"),

    F.sum(F.when(
        F.col("is_mappable") == True
        & ~F.coalesce(valid_location, F.lit(False)),
        1
    ).otherwise(0)).alias("incorrect_map_flags"),

    F.sum(F.when(
        F.col("persons_injured").isNull(), 1
    ).otherwise(0)).alias("missing_injury_counts"),

    F.sum(F.when(
        F.col("persons_killed").isNull(), 1
    ).otherwise(0)).alias("missing_fatality_counts"),

    F.min("crash_date").alias("first_date"),
    F.max("crash_date").alias("last_date")

).first().asDict()

# 4. Print results
print("SMARTCITY AI - ROAD SAFETY QUALITY REPORT")
print("=" * 55)

for key, value in stats.items():
    print(f"{key}: {value}")

source_count = spark.table(SOURCE).count()
print("Reparsed Bronze records:", source_count)

# 5. Final assertions
assert stats["total_collisions"] > 0

assert (
    stats["unique_ids"] ==
    stats["total_collisions"]
), "Duplicate collision IDs!"

for field in [
    "invalid_ids",
    "invalid_dates",
    "invalid_hours",
    "invalid_casualties",
    "incorrect_map_flags"
]:
    assert stats[field] == 0, (
        f"Quality check failed: {field}"
    )

assert stats["total_collisions"] <= source_count

print("\nSUCCESS: ALL ROAD SAFETY QUALITY TESTS PASSED!")


SMARTCITY AI - ROAD SAFETY QUALITY REPORT
total_collisions: 6482
unique_ids: 6482
invalid_ids: 0
invalid_dates: 0
invalid_hours: 0
invalid_casualties: 0
mappable_collisions: 6266
missing_coordinates: 78
incorrect_map_flags: 0
missing_injury_counts: 0
missing_fatality_counts: 0
first_date: 2025-01-01
last_date: 2025-01-31
Reparsed Bronze records: 6482

SUCCESS: ALL ROAD SAFETY QUALITY TESTS PASSED!


In [0]:

# SMARTCITY AI - COLLISIONS BY BOROUGH

borough_stats = (
    df.groupBy("borough_clean")
    .agg(
        F.count("*").alias("total_collisions"),
        F.sum(
            F.when(F.col("is_mappable"), 1)
            .otherwise(0)
        ).alias("mappable_collisions")
    )
    .orderBy(F.desc("total_collisions"))
)

display(borough_stats)


borough_clean,total_collisions,mappable_collisions
BROOKLYN,1870,1810
QUEENS,1361,1329
null,1326,1266
MANHATTAN,879,846
BRONX,814,785
STATEN ISLAND,232,230
